# Imbalanced Bankruptcy Classification

# Introduction

In L1 we turned messy compressed JSON into clean tables and discovered
something important: bankruptcies are **rare**. That single fact — far
more `bankrupt = 0` than `bankrupt = 1` — quietly breaks the most
familiar way of judging a model.

> ❓ If 98% of firms survive, a model that blindly predicts "everyone
> survives" is 98% accurate. Is it a good model?

This notebook builds the vocabulary to answer that honestly.

By the end of this notebook you will be able to:

-   Measure and visualize **class imbalance** (rare bankruptcies
    vs. many non-bankrupt firms).
-   Explain why **accuracy can be misleading** for rare-event
    prediction.
-   Compute and interpret key classification metrics:
    -   confusion matrix
    -   precision, recall, F1
    -   ROC–AUC and Precision–Recall AUC (PR–AUC)
-   Build a first **baseline classifier** (majority-class predictor and
    logistic regression).
-   Prepare for nonlinear models that can capture more complex patterns.

> 📌 **Tip — Data dictionary**
>
> If you want the meaning of each `feat_*` column (and dataset
> structures), open the **Data Dictionary notebook**:
> `data-dictionary.ipynb`.

# 1. Conceptual Foundation

Before we touch a model, we need a shared mental model of *why* rare
events are hard and *which* metrics tell the truth about them. This
section walks from the imbalance problem, through the confusion matrix,
to the full metric vocabulary (precision, recall, F1, ROC–AUC, PR–AUC)
and the idea of an honest baseline.

## Why bankruptcy prediction is usually imbalanced

In many real-world settings, the "positive" outcome is rare:

-   fraud detection
-   equipment failure
-   disease diagnosis
-   **bankruptcy**

In our case, **bankruptcies are rare**, so the dataset will likely have
many more `bankrupt = 0` than `bankrupt = 1`.

> 🧠 That imbalance is not a data-quality problem to "fix" — it is the
> reality of the domain. What it changes is **how we evaluate models**.

➡️ To see why, watch one popular metric fall apart.

## The trap: why accuracy is often misleading

Imagine:

-   98% of companies do not go bankrupt
-   2% go bankrupt

A model that **always predicts "not bankrupt"** gets:

-   **Accuracy = 98%**

> ❗️ **But wait — 98% accuracy catches zero bankruptcies.** The metric
> rewards the model for ignoring exactly the cases we built it to find.

So we need metrics that evaluate *rare-event detection*, not just
overall correctness.

## Confusion matrix (the story behind the numbers)

Every classification metric is built from four counts. For binary
classification we have:

-   **Positive class**: bankrupt (`1`)
-   **Negative class**: not bankrupt (`0`)

|              |         Predicted 0 |         Predicted 1 |
|--------------|--------------------:|--------------------:|
| **Actual 0** |  True Negative (TN) | False Positive (FP) |
| **Actual 1** | False Negative (FN) |  True Positive (TP) |

> 🔍 **Read it in business terms:**
>
> -   `FP`: you flag a **healthy** firm as bankrupt (a false alarm).
> -   `FN`: you **miss** a real bankruptcy — often the most costly error.
> -   `TP`: a real bankruptcy you correctly caught.
> -   `TN`: a healthy firm correctly cleared.

The whole game of imbalanced classification is keeping `FN` low without
drowning in `FP`.

➡️ Those four counts feed the three headline metrics next.

## Precision, recall, F1 — when positives are rare

These metrics focus on the **positive (bankrupt) class** — exactly the
one accuracy ignores.

### Precision

> Of all firms predicted bankrupt, how many truly are bankrupt?

$$\text{precision} = \frac{TP}{TP + FP}$$

High precision means **few false alarms**.

### Recall (Sensitivity)

> Of all truly bankrupt firms, how many did we catch?

$$\text{recall} = \frac{TP}{TP + FN}$$

High recall means **you miss few bankruptcies**.

### F1 score

F1 balances precision and recall (their harmonic mean):

$$F1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}} {\text{precision} + \text{recall}}$$

> ⚠️ F1 is good when you want **one** number, but it hides the trade-off
> — always look at `precision` **and** `recall` too. In bankruptcy,
> `recall` on the positive class is usually the headline.

## Probabilities and thresholds

Many models (like logistic regression) output **probabilities**, not
hard labels.

> 📌 **Note:** Logistic Regression was covered in Project 4 — Predicting
> Earthquake Damage in Nepal (Classification).

To convert a probability `p` into a class prediction, we pick a
**threshold**:

-   if `p >= threshold` → predict `1`
-   else → predict `0`

The default is often `0.5`, but with rare events that may be a poor
choice. Moving the threshold slides us along the precision/recall
trade-off:

| Move threshold | Positives predicted | Recall | Precision |
|---|---|---|---|
| **Lower** | more | ⬆️ higher | ⬇️ lower |
| **Higher** | fewer | ⬇️ lower | ⬆️ higher |

> 🧠 The threshold is a **dial**, not a constant. Section 11 lets you
> turn it and watch the metrics move.

## ROC curve and ROC–AUC

A ROC (Receiver Operating Characteristic) curve shows how a classifier's
performance changes as you vary the decision threshold. It plots the
**true positive rate** (recall) against the **false positive rate**, so
you can see the trade-off between catching more positives and triggering
more false alarms. **ROC–AUC** (Area Under the ROC Curve) summarizes
this curve into a single number between 0 and 1: values closer to 1
indicate better separation between the classes, while 0.5 is roughly
equivalent to random guessing.

ROC curve plots:

-   True Positive Rate (`TPR = recall`)
-   False Positive Rate (`FPR = FP / (FP + TN)`)

As the threshold varies, we trace a curve. **ROC–AUC** summarizes it:

-   `0.5`: random guessing
-   `1.0`: perfect ranking

> ⚠️ ROC–AUC is useful, but with **extreme imbalance** it can look "too
> optimistic" — the huge negative class makes the false-positive rate
> easy to keep small.

## Precision–Recall curve and PR–AUC (often better for rare events)

The Precision–Recall (PR) curve plots **precision vs recall** across
thresholds. For rare events, it is often more informative than ROC,
because it focuses entirely on the positive class.

> 🚦 **ROC–AUC vs PR–AUC — which lens?**
>
> | | ROC–AUC | PR–AUC |
> |---|---|---|
> | Plots | TPR vs FPR | precision vs recall |
> | No-skill baseline | `0.5` (fixed) | **prevalence** (the positive rate) |
> | Under heavy imbalance | can look optimistic | **more honest** |
>
> ✅ **Verdict:** under heavy imbalance, trust **PR–AUC** more.

> 📌 **Baseline for PR–AUC:** the prevalence of the positive class. If
> bankruptcies are 2%, a PR curve hugging precision ≈ 2% is basically
> random.

## Baselines: how to start honestly

Before building complex models, define baselines — the bar every real
model must clear:

1.  **Majority-class predictor**
    -   Always predict `0` (not bankrupt).
    -   Expect very high accuracy, **terrible recall**.

2.  **Logistic regression**
    -   Simple, interpretable.
    -   The first "real" model.
    -   Often surprisingly competitive.

> 🧠 Baselines answer one question: *Are we improving meaningfully, or
> just dressing up the majority-class trick?*

➡️ Watch the short walkthrough video, then practice the metrics on a
tiny hand-built example.

> 🎥 **Walkthrough video**
>
> The video below walks through the imbalanced-classification workflow
> for this lesson. Watch it, then continue with the hands-on sections.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1170265232", h="3298dbabb7", width=700, height=450) 

## Example with dummy data (to practice metrics)

This example uses a tiny dataset to help you understand the metrics
before using the full bankruptcy dataset.

> 🔍 **What to look for:** the snippet hand-codes `2` bankruptcies, with
> **one false alarm and one missed bankruptcy**. Predict the four counts
> (`tn, fp, fn, tp`) before you run it.

**Code 5.2.1.1**:

In [ ]:
import numpy as np

# 2 bankruptcies
y_true = np.array([0, 0, 0, 0, 1, 1])
# one false alarm, one missed bankruptcy
y_pred = np.array([0, 0, 0, 1, 0, 1])

tn = int(((y_true == 0) & (y_pred == 0)).sum())
fp = int(((y_true == 0) & (y_pred == 1)).sum())
fn = int(((y_true == 1) & (y_pred == 0)).sum())
tp = int(((y_true == 1) & (y_pred == 1)).sum())

tn, fp, fn, tp

Now turn those four counts into the three headline metrics by hand —
this is exactly what scikit-learn does for us later.

**Code 5.2.1.2**:

In [ ]:
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall)
    else 0.0
)

precision, recall, f1

### What to notice

-   Even if accuracy is decent, **recall can be low** if you miss
    bankruptcies.
-   **Precision** depends on how many false alarms you create.

So the same predictions can look "fine" or "alarming" depending on which
metric you read — which is why we never read just one.

### How to interpret the metrics (what values are "good"?)

All metrics in this notebook are scaled between **0 and 1**, and for all
of them:

-   **closer to 1 is better**
-   **closer to 0 is worse**

However, each metric answers a *different question*, so "good" depends
on what kind of mistakes you want to avoid (missing bankruptcies
vs. false alarms).

| Metric | Range | Better value | What it measures (intuition) | Common pitfall |
|-------|--:|------|------------------------------|---------------------------|
| **Accuracy** | 0–1 | Higher | Fraction of correct predictions overall | Can look high even if you miss most bankruptcies (imbalance trap) |
| **Precision** | 0–1 | Higher | When the model predicts bankrupt, how often it's right | Can be high while still missing many bankruptcies (low recall) |
| **Recall** | 0–1 | Higher | Of all true bankruptcies, how many the model catches | Can be high by predicting "bankrupt" too often (many false alarms) |
| **F1-score** | 0–1 | Higher | Balance of precision and recall (harmonic mean) | Hides the trade-off: always check precision and recall too |
| **ROC–AUC** | 0–1 | Higher | How well the model ranks positives above negatives across thresholds | Can look "good" even when positive class is extremely rare |
| **PR–AUC** | 0–1 | Higher | Precision–Recall performance across thresholds (often best for rare events) | Baseline depends on prevalence; compare against the positive rate |

#### Two practical anchors for interpretation

1.  **Random / no-skill reference**

    -   For **ROC–AUC**, random guessing ≈ **0.5**
    -   For **PR–AUC**, random guessing ≈ **prevalence** (the fraction
        of bankruptcies)

2.  **What matters most in bankruptcy**

    -   Missing a bankruptcy (**false negative**) is often more costly
        than a false alarm (**false positive**), so **recall** and
        **PR–AUC** usually deserve extra attention.

➡️ With the vocabulary in hand, we apply it to the real Poland dataset.

# Applied Exercises

## 2. Setup

> 📦 We import pandas/numpy, the scikit-learn metrics and models we'll
> need, and — crucially — `wrangle` from `data.py`, the very function we
> built in L1. We don't re-implement loading; we reuse it.

**Code 5.2.2.1**:

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
   accuracy_score,
   average_precision_score,
   classification_report,
   confusion_matrix,
   f1_score,
   precision_recall_curve,
   precision_score,
   recall_score,
   roc_auc_score,
   roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

# local import, this wrangle function is the same one created in
# the lesson 1.
from data import wrangle

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)

> 📌 We will focus on **Poland (full dataset with `bankrupt`)** in this
> notebook. Later, you will apply the same pipeline to Taiwan.

## 3. Load and wrangle the Poland dataset

### Problem

Read data from `data/poland-bankruptcy-data-2009.json.gz`.

### Approach

You should reuse the `wrangle()` function from Notebook 1, which is
already defined in `data.py`. If you need to confirm which dataset
contains the target column, see `data-dictionary.ipynb`.

**Code Task 5.2.3.1**:

In [ ]:
poland_path = ...
poland_df = ...

poland_df.shape

### Checkpoint

> 🧪 Confirms we loaded the **full** Poland file — the one that actually
> has the `bankrupt` label. (The features-only file would silently break
> everything downstream.)

**Code 5.2.3.2**:

In [ ]:
assert "bankrupt" in poland_df.columns, (
    "Expected 'bankrupt' column. "
    "Make sure you are using the *full* Poland file."
)

## 4. Visualize class imbalance

### Problem

Measure how many bankrupt vs. non-bankrupt firms we have.

### Approach

Store the frequency of each distinct class value in the `bankrupt`
column in a variable called `target_counts`.

Similarly, calculate the relative frequency of each class value in the
`bankrupt` column and store it in a variable called `target_rate` (use
`normalize=True`). We exclude missing values here (the default), since
`NaN` is not a class label.

At the end, plot a bar chart from `target_counts`, sort it by index, and
set the title to "Poland: Class counts (bankrupt vs non-bankrupt)", the
x-axis label to "bankrupt", and the y-axis label to "count".

Key points:

-   [value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
-   [plot](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.html)

**Code 5.2.4.1**:

In [ ]:
target_counts = poland_df["bankrupt"].value_counts()
target_counts

Now the same counts as **proportions** — easier to read as a prevalence.

**Code 5.2.4.2**:

In [ ]:
target_rate = poland_df["bankrupt"].value_counts(normalize=True)
target_rate

### Simple visualization

> 🔍 **What to look for:** two bars of wildly different height. The `0`
> bar (survivors) should tower over the `1` bar (bankruptcies).

**Code 5.2.4.3**:

In [ ]:
ax = target_counts.sort_index().plot(kind="bar")  # create `ax` for a bar chart from `target_counts`
ax.set_title("Poland: Class counts (bankrupt vs non-bankrupt)")  # change the ax title
ax.set_xlabel("bankrupt")  # change the ax x-axis label
ax.set_ylabel("count")  # change the ax y-axis label

> 📊 **Reading the bar chart.** The non-bankrupt bar dwarfs the bankrupt
> bar — this is the imbalance from L1 made visual. Keep `target_rate` in
> mind: that small positive proportion is the **PR–AUC no-skill
> baseline** we'll compare against later.

### Checkpoint (imbalance exists)

> 🧪 Asserts there are exactly two classes, both non-empty, and that the
> majority outnumbers the minority by more than 1.5× — i.e., the dataset
> really is imbalanced.

**Code 5.2.4.4**:

**Note: If the dataset is imbalanced, the majority class should be much
larger.**

In [ ]:
assert target_counts.size == 2, (
    "Expected two classes in 'bankrupt'. "
    "Check your dataset and wrangling."
)
assert target_counts.min() > 0, "Both classes should have at least 1 example."
assert target_counts.max() / target_counts.min() > 1.5

## 5. Prepare X (features) and y (target)

### Problem

Usually when we load data from a file, we need to prepare it for use.
The most common things to check are data types and column names, and
sometimes we also need to handle missing values (NaN).

### Approach

-   `y` should be the `bankrupt` column
-   `X` should be all `feat_*` columns
-   We do *not* use the identifier as a feature (`company_id`)

> ⚠️ Leaking the identifier into `X` would let the model "memorize" rows
> instead of learning from financials. Features only.

Key points:

-   [dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html)
-   [copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html)
-   [astype](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html)
-   [startswith](https://docs.python.org/3/library/stdtypes.html#str.startswith)
-   [list
    comprehensions](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions)

**Code Task 5.2.5.1**:

In [ ]:
# drop NaN observations and make a copy of poland_df
poland_df = ...

# select feature columns that starts with "feat_"
feature_cols = [ ... ]

X = ...
y = ...  # also force the value as integer type.

X.shape, y.shape

### Checkpoint

> 🧪 Confirms `X` has the **64** Poland features and `y` contains only
> `{0, 1}` — the shape every scikit-learn classifier expects.

**Code 5.2.5.2**:

In [ ]:
assert X.shape[1] == 64, "Poland should have feat_1..feat_64"
assert set(np.unique(y)).issubset({0, 1})

## 6. Train/test split

### Problem

To train and test our ML model, we need to split `X` and `y` into
training and test sets.

### Approach

Create training and test sets.

> 💡 **Why `stratify=y`?** Because the data is imbalanced, a random split
> could land very few bankruptcies in the test set (or none). Stratifying
> keeps the class proportions **the same** in train and test, so the
> evaluation is trustworthy.

Key points:

-   [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

**Code Task 5.2.6.1**:

In [ ]:
X_train, X_test, y_train, y_test = ...

X_train.shape, X_test.shape

### Checkpoint

> 🧪 Asserts the bankruptcy **rate** in train and test differ by less
> than 1% — proof that stratification preserved the imbalance on both
> sides.

**Note: The rates should be reasonably close (not identical, but
close).**

**Code 5.2.6.2**:

In [ ]:
train_rate = y_train.mean()
test_rate = y_test.mean()

assert abs(train_rate - test_rate) < 0.01

train_rate, test_rate

## 7. Baseline #1: Majority-class predictor

### Problem

Before using more complex models, we can start with a simple baseline
model (`DummyClassifier`) to use as a reference.

### Approach

Start with a simple baseline that predicts the most frequent class.

Fit a simple baseline classifier using `X_train` and `y_train`. Then
generate predictions for `X_test` and store them in `y_pred_dummy`.
Next, compute the baseline metrics and store them in `acc_dummy`,
`prec_dummy`, `rec_dummy`, and `f1_dummy` (use `zero_division=0`).
Finally, compute the confusion matrix and store it in `cm_dummy`.

> 🔍 **Predict before you run:** a "always say 0" model will post high
> accuracy and **recall = 0**. Watch for exactly that.

Key points:

-   [DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html)
-   [fit](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html#sklearn.dummy.DummyClassifier.fit)
-   [predict](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html#sklearn.dummy.DummyClassifier.predict)
-   [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
-   [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
-   [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
-   [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
-   [confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

**Code Task 5.2.7.1**:

In [ ]:
dummy = ...  # use DummyClassifier with most frequent strategy
# train the dummy model on X_train and y_train
...

y_pred_dummy = dummy.predict(X_test)

### Evaluate baseline

**Code 5.2.7.2**:

In [ ]:
acc_dummy = accuracy_score(y_test, y_pred_dummy)
prec_dummy = precision_score(y_test, y_pred_dummy, zero_division=0)
rec_dummy = recall_score(y_test, y_pred_dummy, zero_division=0)
f1_dummy = f1_score(y_test, y_pred_dummy, zero_division=0)

acc_dummy, prec_dummy, rec_dummy, f1_dummy

### Confusion matrix

**Code 5.2.7.3**:

In [ ]:
# create a confusion matrix between y_test and y_pred_dummy
cm_dummy = confusion_matrix(y_test, y_pred_dummy)
cm_dummy

> 📊 **Reading the dummy's confusion matrix.** Its entire second column
> is zero — it never predicts bankrupt. Every real bankruptcy lands in
> `FN`. That's why its **recall is exactly 0** even as accuracy looks
> excellent: the textbook imbalance trap.

### Checkpoint (accuracy high but recall low)

> 🧪 The assert pins down the lesson: `rec_dummy == 0.0`. High accuracy,
> zero bankruptcies caught.

**Note: This is the key lesson: accuracy can look good even if recall is
terrible.**

**Code 5.2.7.4**:

In [ ]:
assert rec_dummy == 0.0

## 8. Baseline #2: Logistic regression

### Problem

We can now check the results with a Logistic Regression model — our
first "real" classifier.

### Approach

Create a logistic regression model and set `max_iter=1000` and
`solver="lbfgs"`. Fit the model using `X_train` and `y_train`. Then
generate class predictions for `X_test` and store them in `y_pred_lr`.
Also compute predicted probabilities for the positive class and store
them in `y_proba_lr`. Next, evaluate the model by computing `acc_lr`,
`prec_lr`, `rec_lr`, and `f1_lr` (use `zero_division=0`). Compute the
confusion matrix and store it in `cm_lr`. After that, use `y_proba_lr`
to compute `roc_auc` and `pr_auc`. Print a classification report with
`digits=3` and `zero_division=0` to summarize performance. Finally, run
the checkpoint to confirm that `y_pred_lr` is not identical to
`y_pred_dummy`.

Key points:

-   [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
-   [fit](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.fit)
-   [predict](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict)
-   [predict_proba](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba)
-   [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
-   [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
-   [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
-   [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
-   [confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
-   [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
-   [average_precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)
-   [classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)

**Code 5.2.8.1**:

In [ ]:
# instantiate the logistic regression model with `max_iter=1000` and `solver="lbfgs"`
log_reg = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
)

log_reg.fit(X_train, y_train)  # train log_reg with X_train and y_train

**Code 5.2.8.2**:

In [ ]:
# use log_reg to predict on X_test
y_pred_lr = log_reg.predict(X_test)
# use `log_reg` to predict probabilities on `X_test`, and select the
# probability of class 1 with `[:, 1]`
y_proba_lr = log_reg.predict_proba(X_test)[:, 1]

y_pred_lr[:5], y_proba_lr[:5]

### Core classification metrics

**Code 5.2.8.3**:

In [ ]:
# compute metrics between `y_test` and `y_pred_lr`
acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr, zero_division=0)
rec_lr = recall_score(y_test, y_pred_lr, zero_division=0)
f1_lr = f1_score(y_test, y_pred_lr, zero_division=0)

acc_lr, prec_lr, rec_lr, f1_lr

### Confusion matrix

**Code 5.2.8.4**:

In [ ]:
# compute the confusion matrix between `y_test` and `y_pred_lr`
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_lr

> 📊 **Compare to the dummy.** Look at the bottom row (`Actual 1`): does
> logistic regression move *any* real bankruptcies out of `FN` and into
> `TP`? Even a small bump in recall over the dummy's zero is the first
> sign of a model that's actually learning.

### ROC–AUC and PR–AUC (use probabilities)

> 🔍 These two use **probabilities** (`y_proba_lr`), not hard labels, so
> they judge the model across *all* thresholds at once. Remember the
> no-skill anchors: `0.5` for ROC–AUC, **prevalence** for PR–AUC.

**Code 5.2.8.5**:

In [ ]:
# calculate ROC-AUC using `y_test` and `y_proba_lr`
roc_auc = roc_auc_score(y_test, y_proba_lr)
# calculate the average precision score using `y_test` and `y_proba_lr`
pr_auc = average_precision_score(y_test, y_proba_lr)

roc_auc, pr_auc

### Classification report (summary table)

**Code 5.2.8.6**:

In [ ]:
# call the classification_report for `y_test`, `y_pred_lr`, `digits=3`
# and `zero_division=0`
classification_report(
    y_test,
    y_pred_lr,
    digits=3,
    zero_division=0,
)

### Checkpoint

> 🧪 Confirms logistic regression's predictions actually **differ** from
> the dummy's — i.e., it learned something beyond "always 0".

**Note: Logistic regression should usually do better than predicting
only the majority class.**

**Code 5.2.8.7**:

In [ ]:
assert (y_pred_lr != y_pred_dummy).any()

## 9. Compare metrics with and without feature scaling (MinMaxScaler)

### Problem

Logistic regression learns a linear boundary using feature coefficients.
If features are on very different scales, optimization can be harder and
the model may weight features differently than expected.

In our datasets, `feat_*` columns are ratios, but their ranges can still
differ a lot (and some may contain extreme values). Scaling is one of
the simplest preprocessing steps to test.

### Approach

We will compare:

1.  Logistic Regression **without** scaling
2.  Logistic Regression **with** `MinMaxScaler`

> 💡 **Why `MinMaxScaler` in a `Pipeline`?** A pipeline fits the scaler
> on the **training data only**, then reuses that transform on the test
> data — preventing the test set from leaking into preprocessing. It also
> packages scaler + model into one object you can `fit`/`predict` as a
> unit.

> 📌 **Note:** We use `MinMaxScaler` as an example. Other common scalers
> you may want to explore later:
>
> -   `StandardScaler` (centers and scales to unit variance)
> -   `RobustScaler` (uses median/IQR, more robust to outliers)
> -   `MaxAbsScaler` (scales by max absolute value)

Key points:

-   [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
-   [fit](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.fit)
-   [predict](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict)
-   [predict_proba](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba)
-   [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
-   [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
-   [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
-   [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
-   [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
-   [average_precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)
-   [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
-   [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)
-   [confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

### Helper: evaluate a classifier using the same metrics

> 📦 `evaluate_binary_classifier` bundles all six metrics into one dict so
> every model is scored **identically** — a fair-comparison habit we lean
> on heavily in L4.

**Code 5.2.9.1**:

In [ ]:
def evaluate_binary_classifier(
    *,
    name: str,
    y_true: np.ndarray | pd.Series,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
) -> dict[str, float | str]:
    """
    Compute key metrics for binary classification.

    Parameters
    ----------
    name
        Label for the model (used in the results table).
    y_true
        True labels (0/1).
    y_pred
        Predicted labels (0/1).
    y_proba
        Predicted probabilities for class 1.

    Returns
    -------
    dict of str to float | str
        Metrics: accuracy, precision, recall, f1, roc_auc, pr_auc.
    """
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
    }

### Logistic Regression (no scaling)

**Code 5.2.9.2**:

In [ ]:
log_reg_plain = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
)
log_reg_plain.fit(X_train, y_train)

y_pred_plain = log_reg_plain.predict(X_test)
y_proba_plain = log_reg_plain.predict_proba(X_test)[:, 1]

### Logistic Regression + MinMaxScaler (pipeline)

Important idea: we fit the scaler **only on the training data**, then
apply the same transformation to the test data. `Pipeline` makes this
safe and simple.

**Code Task 5.2.9.3**:

In [ ]:
# create a pipeline log_reg_scaled with 2 steps: MinMaxScaler as scaler and
# LogisticRegression as model (using the same parameters used in Code 5.2.9.2)
log_reg_scaled = ...
# train `log_reg_scaled` using `X_train` and `y_train`
...

# generate predictions for `X_test` using `log_reg_scaled`
y_pred_scaled = ...
# predict probabilities for `X_test` using `log_reg_scaled`,
# and select the class 1 probabilities with `[:, 1]`
y_proba_scaled = ...

In [ ]:
# create a pipeline log_reg_scaled with 2 steps: MinMaxScaler as scaler and
# LogisticRegression as model (using the same parameters used in Code 5.2.9.2)
log_reg_scaled = Pipeline(
    steps=[
        ("scaler", MinMaxScaler()),
        ("model", LogisticRegression(max_iter=1000, solver="lbfgs")),
    ]
)
# train `log_reg_scaled` using `X_train` and `y_train`
log_reg_scaled.fit(X_train, y_train)

# generate predictions for `X_test` using `log_reg_scaled`
y_pred_scaled = log_reg_scaled.predict(X_test)
# predict probabilities for `X_test` using `log_reg_scaled`,
# and select the class 1 probabilities with `[:, 1]`
y_proba_scaled = log_reg_scaled.predict_proba(X_test)[:, 1]

### Compare metrics side-by-side

> 🔍 **What to look for:** read **down the `recall` and `pr_auc`
> columns**, not the `accuracy` column. Those are the ones that matter on
> imbalanced data.

**Code 5.2.9.4**:

In [ ]:
rows = [
    evaluate_binary_classifier(
        name="LogReg (no scaling)",
        y_true=y_test,
        y_pred=y_pred_plain,
        y_proba=y_proba_plain,
    ),
    evaluate_binary_classifier(
        name="LogReg + MinMaxScaler",
        y_true=y_test,
        y_pred=y_pred_scaled,
        y_proba=y_proba_scaled,
    ),
]

results = pd.DataFrame(rows).set_index("model")
results

> 📊 **Reading the comparison table.** If scaling barely moves `recall`
> and `pr_auc`, that tells you these particular features didn't need it —
> not that scaling is useless in general. The habit of scoring both
> models the same way is the real takeaway.

### Compare confusion matrices

**Code 5.2.9.5**:

In [ ]:
# generate the confusion matrix using `y_test` and `y_pred_plain`
cm_plain = confusion_matrix(y_test, y_pred_plain)
# generate the confusion matrix using `y_test` and `y_pred_scaled`
cm_scaled = confusion_matrix(y_test, y_pred_scaled)

cm_plain, cm_scaled

### Checkpoint (did we actually run two different pipelines?)

> 🧪 This checkpoint only confirms both pipelines **ran** and produced
> finite probabilities of the right length — it deliberately does **not**
> assert that scaling improved anything.

**Code 5.2.9.6**:

**Notes:**

-   The point is not that scaling MUST improve performance every time.
-   The checkpoint just ensures both pipelines ran and produced
    probabilities.

In [ ]:
assert len(y_proba_plain) == len(y_test)
assert len(y_proba_scaled) == len(y_test)
assert np.isfinite(y_proba_plain).all()
assert np.isfinite(y_proba_scaled).all()

### What should you conclude?

> 🧠 **Key insight:** scaling is a *test*, not a guarantee.

-   Sometimes scaling helps logistic regression, sometimes it changes
    little.

-   If scaling changes results, it's a sign that:

    -   feature ranges differ meaningfully, and/or
    -   the solver benefits from more uniform feature magnitudes.

> ⚠️ **An important imbalance subtlety.** Logistic regression may assign
> probabilities below `0.5` to **all** firms (because bankruptcies are
> rare and the model is conservative), so it predicts only the majority
> class (`0`) on the test set. When a model predicts no positives:
>
> -   `TP = 0` → **recall = 0** (you caught no bankruptcies)
> -   precision is defined as `0` (no predicted positives; libraries
>     return `0` to avoid division-by-zero) → **F1 = 0**
>
> MinMax scaling doesn't change this if the ranking/probabilities still
> don't cross `0.5`. Scaling can help *optimization*, but it doesn't
> guarantee any probability exceeds the default threshold. The fixes:
> **inspect the predicted-probability distribution, lower the
> threshold** (§11), and/or use **class weights / resampling** (you'll
> explore resampling in L3).

➡️ That last point — that the threshold itself is a lever — is exactly
what the ROC and PR curves visualize next.

## 10. Plot ROC curve and Precision–Recall curve

### Problem

When classes are imbalanced, a single metric like accuracy can hide how
the model behaves, so we use ROC and Precision–Recall curves to evaluate
performance across different decision thresholds and understand the
trade-offs between catching bankruptcies and raising false alarms.

### Approach

First, use `y_test` and `y_proba_lr` to compute the ROC curve values and
store them in `fpr`, `tpr`, and `roc_thresholds`. Then build a small
DataFrame with `fpr` and `tpr` and plot it, setting the title and axis
labels as shown. Next, compute the Precision–Recall curve values and
store them in `prec_curve`, `rec_curve`, and `pr_thresholds`. Plot
precision against recall using a DataFrame, and set the title and axis
labels. Finally, compute the prevalence of the positive class by taking
the mean of `y_test` and store it in `prevalence` so you can compare it
to the PR curve as a baseline reference.

Key points:

-   [roc_curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html)
-   [plot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html)
-   [precision_recall_curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html)

### ROC Curve

**Code 5.2.10.1**:

In [ ]:
# compute the ROC curve from `y_test` and `y_proba_lr`
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba_lr)

ax = pd.DataFrame({"fpr": fpr, "tpr": tpr}).plot(x="fpr", y="tpr")
ax.set_title("ROC curve (Logistic Regression)")
ax.set_xlabel("False Positive Rate (FPR)")
ax.set_ylabel("True Positive Rate (TPR / Recall)")

> 📊 **Reading the ROC curve.** It bows toward the top-left corner the
> better the model ranks bankruptcies above survivors. The diagonal is
> random guessing (ROC–AUC = `0.5`). Remember the caveat: under heavy
> imbalance this curve can look flatteringly good.

### Precision–Recall curve

> 🔍 The honest lens for rare events. Watch where precision sits as you
> push recall higher.

**Code 5.2.10.2**:

In [ ]:
# compute the Precision–Recall curve from `y_test` and `y_proba_lr`
prec_curve, rec_curve, pr_thresholds = precision_recall_curve(
    y_test, y_proba_lr
)

ax = pd.DataFrame({"recall": rec_curve, "precision": prec_curve}).plot(
    x="recall",
    y="precision",
)
ax.set_title("Precision–Recall curve (Logistic Regression)")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")

### A useful reference line (prevalence)

The baseline precision for a random classifier is the fraction of
positives:

**Code 5.2.10.3**:

In [ ]:
prevalence = float(y_test.mean())
prevalence

> 📊 If the PR curve stays close to `prevalence`, the model is close to
> random. The gap between the curve and this prevalence line is the real
> signal of skill on the rare class.

## 11. Threshold exploration (optional but recommended)

In this section, you can experiment with the `threshold` variable by
changing its value to see how predictions change as you move the
threshold.

> 🚦 **Turn the dial.** Try `0.5`, then `0.2`, then `0.1`. Lowering the
> threshold predicts more positives → **recall climbs, precision usually
> falls.** This is the precision/recall trade-off from §1, now hands-on.

Key points:

-   [astype](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html)
-   [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
-   [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
-   [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
-   [set](https://docs.python.org/3/library/stdtypes.html#set)
-   [numpy.unique](https://numpy.org/doc/stable/reference/generated/numpy.unique.html)
-   [set.issubset](https://docs.python.org/3/library/stdtypes.html#set.issubset)

**Code 5.2.11.1**:

In [ ]:
threshold = 0.2  # try 0.5, 0.2, 0.1 ...
y_pred_thr = (y_proba_lr >= threshold).astype(int)

precision_thr = precision_score(y_test, y_pred_thr, zero_division=0)
recall_thr = recall_score(y_test, y_pred_thr, zero_division=0)
f1_thr = f1_score(y_test, y_pred_thr, zero_division=0)

precision_thr, recall_thr, f1_thr

### Checkpoint

> 🧪 Confirms the thresholded predictions are well-formed (right shape,
> only `{0, 1}`, metrics in `[0, 1]`) for whatever threshold you chose.

**Code 5.2.11.2**:

In [ ]:
# Lowering threshold often increases recall (but may reduce precision).
assert y_pred_thr.shape == y_test.shape
assert set(np.unique(y_pred_thr)).issubset({0, 1})
assert 0.0 <= precision_thr <= 1.0
assert 0.0 <= recall_thr <= 1.0
assert 0.0 <= f1_thr <= 1.0

## 12. Apply the same pipeline to Taiwan (Optional/Ungraded)

In this section, you can apply what you learned in the previous sections
to analyze the Taiwan dataset. This section is optional and ungraded,
but it is highly recommended to reinforce what you have learned so far.

> 🧠 Taiwan has **95** features instead of 64, but the workflow is
> identical: `wrangle` → split (stratified) → baselines → metrics →
> curves. Reusing the same steps on new data is the whole point.

In [ ]:
# your code here

------------------------------------------------------------------------

# Wrap-up

In this notebook you learned how to work with **imbalanced
classification** problems using the bankruptcy datasets:

-   You measured imbalance and saw why it is expected for rare events.

-   You learned why **accuracy alone is misleading**.

-   You computed and interpreted:

    -   confusion matrix
    -   precision, recall, F1
    -   ROC–AUC and PR–AUC

-   You trained two baselines:

    -   majority-class predictor (useful, but often fails on recall)
    -   logistic regression (first real baseline model)

> 🧠 **The thread to carry forward:** on rare-event data, **recall on the
> bankrupt class and PR–AUC** are the metrics that tell the truth — and a
> baseline is the bar every fancier model must beat.

➡️ **Next:** logistic regression draws a single linear boundary, which
misses complex interactions between financial indicators. In the next
notebook we'll explore **nonlinear ensemble models** (Random Forests)
that capture those interactions — and we'll compare them *fairly*
against these baselines.